In [1]:
import os
import pickle

import numpy as np 
import pandas as pd 

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from scipy import stats
import scipy.optimize as optimize 

In [2]:
# Reading the saved data pickle file
df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)
df["VIX"] = df["adj close"].apply(np.int)


In [3]:
org_data = pd.read_csv("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/csv/optionsTradeData.csv")

In [4]:
# Calculate return

vix_yesterday = df['VIX'].shift(periods=1)  # Series made up of shifted KOSPI200
np
return_array=np.log(df['VIX']/vix_yesterday)
#return_array = (df['VIX']-VIX_yesterday) / VIX_yesterday  # Calculate return

In [5]:
# GARCH model
def garch_forward(return_rate,variance,coefficients):
    ''' data type: float, float, 1d array(length=3)'''
    
    # Coefficients
    alpha,beta,omega = coefficients
    # Calculate
    return omega + alpha*return_rate*return_rate + beta*variance

In [6]:
# Set VKOSPI as an initial variance in the model

initial_vix = df['VIX'][1] 
initial_variance = initial_vix*initial_vix/2520000

In [7]:
# Function for optimization

def garch_for_optimization(array): 
    ''' data type: 1d array(length=3)'''
    
    # Coeffcients
    alpha,beta,omega = array 
    
    # Variables
    sum_probability = 0 # to maximize
    variance = initial_variance
    
    for i in range(1,return_array_train.shape[0]):  # exclude the first value because it's nan.
        return_rate = return_array_train[i]  
        
        # in case something goes wrong
        if variance<=0:
            print("Negative variance")
            break
        
        # calculate probability in a single day
        probability = -np.log(variance) - return_rate * return_rate / variance
        
        # add to the sum
        sum_probability += probability
        
        # calculate next day's variance by GARCH
        variance = garch_forward(return_rate,variance,array)
   
    return -sum_probability # note the sign(-); because scipy.optimize requires a function to be minimized

In [8]:
# Optimize
# if execute is set TRUE, optimization which takes some time will be initiated
execute = True    

if execute:
    bounds = optimize.Bounds([0,0,0],[1,1,np.inf])
    initial_guess = [0.14,0.76,2.97]  # V-Lab's estimate

    # Trust-constr performed the best 
    optimize_res_trust = optimize.minimize(garch_for_optimization,initial_guess,method='trust-constr',bounds=bounds)

    # Pickle the result since it takes too much time
    with open('../input/optimize_res_trust.pkl','wb') as file:
        pickle.dump(optimize_res_trust,file)

NameError: name 'return_array_train' is not defined

In [9]:
# Load the pickle: optimize_res_trust
with open('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/garch/optimize_res_trust.pkl','rb') as file:
    optimize_res_trust = pickle.load(file)

# Optimization result
print('optimization result:')
print('alpha =',optimize_res_trust.x[0])
print('beta =',optimize_res_trust.x[1])
print('omega =',optimize_res_trust.x[2])

FileNotFoundError: [Errno 2] No such file or directory: '/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/garch/optimize_res_trust.pkl'

In [ ]:
# Load the pickle: optimize_res_trust
with open('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/garch/optimize_res_trust.pkl','rb') as file:
    optimize_res_trust = pickle.load(file)

# Optimization result
print('optimization result:')
print('alpha =',optimize_res_trust.x[0])
print('beta =',optimize_res_trust.x[1])
print('omega =',optimize_res_trust.x[2])